# Stage 1 — Anomaly Detection QC

Steps through each detector individually so outputs can be inspected before running the full pipeline.

| Step | Function | What to check |
|------|----------|---------------|
| 0 | Config | Date windows, optional store filter |
| 1 | Session + data | Row count, schema |
| 2 | Data load | OSA + hierarchy joined, cached |
| 3 | WOW detector | Top signals by LSV |
| 4 | VS_CHAIN detector | Worst chain gaps |
| 5 | REPEAT detector | Most persistent items |
| 6 | TICK_AND_FLICK detector | Stores with rising compliance issues |
| 7 | WMD data load | Markdown log, sales, wastage, item/store hierarchy |
| 8 | MARKDOWN_OVERRIDE VS_CHAIN | Stores overriding above chain rate |
| 9 | MARKDOWN_OVERRIDE WOW | Stores with rising override rate |
| 10 | WASTE VS_CHAIN | Stores with straight-to-bin rate above chain |
| 11 | WASTE WOW | Stores with rising straight-to-bin rate |
| 12 | MARKDOWN_SELLTHROUGH VS_CHAIN | Stores with sell-through below chain |
| 13 | MARKDOWN_SELLTHROUGH WOW | Stores with falling sell-through rate |
| 14 | MARKDOWN_SKU REPEAT_REQUEST | SKUs repeatedly flagged for markdown |
| 15 | MARKDOWN_SKU EXCESSIVE_OVERRIDE | SKUs with high per-item override rate |
| 16 | Merged signals | Trigger breakdown, multi-trigger count |

> **Note:** Steps 7–15 require access to the WMD tables in `ANALYSIS.FINAL`.
> Set `RUN_WMD = True` in the config cell to enable them.

In [35]:
# ── Date window ───────────────────────────────────────────────────────────────
ANALYSIS_DATE_STR = '2025-06-14'   # last day of THIS week

# ── Client config (None = base defaults; 'eastofengland' = client override) ────
CLIENT = None   # or 'eastofengland'

# ── Optional: restrict to a single store for faster QC runs ──────────────────
QC_STORE_ID = None   # e.g. 507 — set to None for all stores

# ── Steps 7–15: WMD detectors — requires WMD tables in ANALYSIS.FINAL ────────
RUN_WMD = True   # set True if you have access to WMD tables

import datetime
analysis_date   = datetime.date.fromisoformat(ANALYSIS_DATE_STR)
THIS_WEEK_END   = str(analysis_date)
THIS_WEEK_START = str(analysis_date - datetime.timedelta(days=6))
LAST_WEEK_END   = str(analysis_date - datetime.timedelta(days=7))
LAST_WEEK_START = str(analysis_date - datetime.timedelta(days=13))
LOOKBACK_START  = str(analysis_date - datetime.timedelta(days=69))  # ~10 weeks
WMD_LOOKBACK    = str(analysis_date - datetime.timedelta(days=27))  # 4 weeks for SKU detectors

print(f'This week  : {THIS_WEEK_START} – {THIS_WEEK_END}')
print(f'Last week  : {LAST_WEEK_START} – {LAST_WEEK_END}')
print(f'Lookback   : {LOOKBACK_START} – {THIS_WEEK_END}')
print(f'WMD window : {WMD_LOOKBACK} – {THIS_WEEK_END}')
print(f'Client     : {CLIENT or "base defaults"}')
print(f'Store filter: {QC_STORE_ID}')
print(f'WMD detectors: {"ENABLED" if RUN_WMD else "SKIPPED"}')

This week  : 2025-06-08 – 2025-06-14
Last week  : 2025-06-01 – 2025-06-07
Lookback   : 2025-04-06 – 2025-06-14
WMD window : 2025-05-18 – 2025-06-14
Client     : base defaults
Store filter: None
WMD detectors: ENABLED


In [36]:
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.pipeline.config.loader import load_config
from store_insight.pipeline.pipeline.session import create_session

cfg     = load_config(CLIENT)
print(f'Config loaded: client={cfg.client_name}')

Config loaded: client=default


In [37]:
session = create_session(cfg)

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/93a31d5d-1a5d-433f-baa0-138549cd2108/saml2?SAMLRequest=nZLRbtowFIZfJfKuE9sJsGABFSurxkS7rASmcmdiAxaJnfo4TdunnwlF6i7ai91EkfOdk8%2FnP6Or56oMnqQFZfQY0YigQOrCCKX3Y7TKb8IUBeC4Frw0Wo7RiwR0NRkBr8qaTRt30PfysZHgAt9IA%2Bs%2BjFFjNTMcFDDNKwnMFWw5vV2wOCKstsaZwpToXcnnFRxAWucNLyUClNc7OFczjNu2jdokMnaPY0IIJkPsqRPy5cI%2F%2Bzt9wFNMeifeEx7P3ty%2BKX0ewWda2zME7EeeZ2H2a5mjYHpRvTYamkrapbRPqpCr%2B8VZALwBVHSQkF7UHEMwfoYRf22sjECbdlfyoyxMVTfOt478G95JgUuzV%2F7289kY1Uclvj6um%2FVmuy4WW2HW0z919lvfwSpuHm7toeKrdGPm6Xf98DM3BQrWl3jjU7xzgEbO9SlU549IPAhJP6RpThOWEEZJNBikGxTMfKhKc9dVXsw7j6hShTVgds7oUmnZWQ4TnlDRFyHl%2FtFLkl245ZyENEn7vWEhYkpSfIo6Ruf1YZ2InfzXUEb4fYu3dbzzCc1nmSlV8RLcGFtx93GANKLdiRLhrkOZrLgqp0JYCeCDLEvTXlvJnd96ZxuJ8OT813%2F3fvIX&RelaySt

 pip install snowflake-connector-python[secure-local-storage]


In [38]:
import snowflake.snowpark.functions as F
hierarchy = session.table('ANALYSIS.FINAL.LSM_ITEM_HIERARCHY')
markdown_sales = session.table('INBOUND.SOURCE.SOURCE_WMD_MARKDOWN_SALES')
hierarchy.show()

markdown_sales.filter(F.col('STORE_ID') == 408).join(hierarchy, on='ITEM_ID', how='left').select('ITEM_ID', 'STORE_ID', 'DEPARTMENT_NAME').show()

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"MASTER_TRACKING_GROUP_ID"  |"ITEM_ID"  |"ITEM_NAME"                |"DIVISION_ID"  |"DIVISION_NAME"         |"DEPARTMENT_ID"  |"DEPARTMENT_NAME"     |"PRODUCT_PROFILE_GROUP_ID"  |"PRODUCT_PROFILE_GROUP_NAME"  |"NEW_ITEM_ID"  |"COLOUR_DESCRIPTION"  |"SIZE_DESCRIPTION"  |"ITEM_DISPLAY_NAME"              |"TRAITED_FLAG"  |"RETAIL_PRICE"  |"UNIVERSAL_PRODUCT_CODE"  |"MASTER_ITEM_ID"  |"SUBCLASS_ID"  |"SUBCLASS_NAME"  |"DSD_FLAG"  |"FILENAME"                           

In [29]:
from snowflake.snowpark import functions as F

# ── Resolve table names from config ──────────────────────────────────────────
OSA_TABLE       = cfg.tables.osa
HIERARCHY_TABLE = cfg.tables.hierarchy
FEEDBACK_TABLE  = cfg.tables.feedback

print(f'OSA table      : {OSA_TABLE}')
print(f'Hierarchy table: {HIERARCHY_TABLE}')
print(f'Feedback table : {FEEDBACK_TABLE}')

osa_raw = (
    session.table(OSA_TABLE)
    .filter(F.col('ANALYSIS_DATE') >= LOOKBACK_START)
    .filter(F.col('ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('STORE_ID').alias('STORE_ID'),
        F.col('ITEM_CODE').alias('ITEM_CODE'),
        F.col('ACTUAL_SALES_VOLUME'),
        F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'),
        F.col('ANALYSIS_DATE'),
        F.col('DEPARTMENT_ID'),
    )
)

item = (
    session.table(HIERARCHY_TABLE)
    .select(
        F.col('ITEM_CODE'),  # HIERARCHY_V1 already aliases MASTER_ITEM_ID as ITEM_CODE
        F.col('ITEM_ID'),
        F.col('DEPARTMENT_NAME'),
        F.col('SUBCLASS_NAME'),
        F.col('PRODUCT_PROFILE_GROUP_NAME'),
    )
    .distinct()
)

osa = (
    osa_raw.join(item, on=['ITEM_CODE'], how='left')
    .filter(F.col('DEPARTMENT_NAME').is_not_null())
    .cache_result()
)

feedback = (
    session.table(FEEDBACK_TABLE)
    .filter(F.col('ALERT_ANALYSIS_DATE') >= LAST_WEEK_START)
    .filter(F.col('ALERT_ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('STORE_ID').alias('STORE_ID'),
        F.col('SITE_NAME'),
        F.col('ALERT_TYPE_NAME'),
        F.col('ALERT_ANALYSIS_DATE'),
        F.col('INTERVENTION_ID'),
        F.col('TICK_AND_FLICK_FLAG'),
    )
    .cache_result()
)

qc_data = osa.filter(F.col('STORE_ID') == QC_STORE_ID) if QC_STORE_ID else osa

print(f'\nOSA rows loaded & cached      : {osa.count():,}')
print(f'Feedback rows loaded & cached : {feedback.count():,}')
print(f'QC data rows                  : {qc_data.count():,}')

OSA table      : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.OSA_V1
Hierarchy table: SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.HIERARCHY_V1
Feedback table : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT.FEEDBACK_V1

OSA rows loaded & cached      : 73,588,943
Feedback rows loaded & cached : 22,275
QC data rows                  : 73,588,943


## Step 3 — WOW Detector

In [32]:
from store_insight.pipeline.stages._1_detect.detectors.osa.wow import detect as detect_wow

wow_signals = detect_wow(
    qc_data,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
)

print(f'WOW signals: {len(wow_signals)}')
for sig in sorted(wow_signals, key=lambda x: x['this_week_lsv'], reverse=True)[:10]:
    print(f"  store={sig['store_id']:>4}  dept={sig['department']:<30}  "
          f"LW={sig['last_week_osa']:.1f}%  TW={sig['this_week_osa']:.1f}%  "
          f"drift={sig['wow_drift_pp']:+.1f}pp  lsv=£{sig['this_week_lsv']:,.2f}")

WOW signals: 224
  store= 239  dept=CIGARETTES                      LW=84.0%  TW=66.5%  drift=-17.4pp  lsv=£3,590.43
  store= 508  dept=SOFT DRINKS                     LW=88.4%  TW=77.0%  drift=-11.5pp  lsv=£1,933.14
  store= 470  dept=SOFT DRINKS                     LW=78.0%  TW=67.3%  drift=-10.7pp  lsv=£1,797.98
  store= 239  dept=TOBACCO                         LW=88.3%  TW=72.4%  drift=-15.9pp  lsv=£1,435.75
  store= 292  dept=FRUIT                           LW=95.8%  TW=92.5%  drift=-3.3pp  lsv=£1,417.84
  store= 508  dept=BEER/LAGER & CIDER              LW=92.0%  TW=81.1%  drift=-10.9pp  lsv=£1,388.15
  store= 433  dept=SOFT DRINKS                     LW=87.0%  TW=85.8%  drift=-1.1pp  lsv=£1,373.43
  store= 508  dept=CIGARETTES                      LW=93.6%  TW=81.9%  drift=-11.8pp  lsv=£1,291.25
  store= 476  dept=CIGARETTES                      LW=85.7%  TW=74.8%  drift=-11.0pp  lsv=£1,208.71
  store= 492  dept=CONFECTIONERY                   LW=82.9%  TW=79.7%  drift=-3.2pp  

## Step 4 — VS_CHAIN Detector

In [33]:
from store_insight.pipeline.stages._1_detect.detectors.osa.vs_chain import detect as detect_vs_chain

vs_chain_signals = detect_vs_chain(
    qc_data,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
)

print(f'VS_CHAIN signals: {len(vs_chain_signals)}')
for sig in sorted(vs_chain_signals, key=lambda x: x['chain_gap_pp'])[:10]:
    print(f"  store={sig['store_id']:>4}  dept={sig['department']:<30}  "
          f"store={sig['this_week_osa']:.1f}%  chain={sig['chain_this_week_osa']:.1f}%  "
          f"gap={sig['chain_gap_pp']:+.1f}pp  lsv=£{sig['this_week_lsv']:,.2f}")

VS_CHAIN signals: 252
  store= 234  dept=IN-STORE BAKERY                 store=13.3%  chain=87.7%  gap=-74.4pp  lsv=£1,036.12
  store= 427  dept=ICE CREAM & RELATED PRODS       store=38.5%  chain=89.3%  gap=-50.9pp  lsv=£785.52
  store= 448  dept=ICE CREAM & RELATED PRODS       store=45.1%  chain=89.3%  gap=-44.2pp  lsv=£568.85
  store= 234  dept=ICE CREAM & RELATED PRODS       store=50.0%  chain=89.3%  gap=-39.3pp  lsv=£1,062.08
  store= 215  dept=ICE CREAM & RELATED PRODS       store=50.8%  chain=89.3%  gap=-38.5pp  lsv=£985.70
  store= 422  dept=MILK                            store=61.1%  chain=92.6%  gap=-31.5pp  lsv=£358.60
  store= 461  dept=ICE CREAM & RELATED PRODS       store=60.2%  chain=89.3%  gap=-29.1pp  lsv=£319.34
  store= 421  dept=TOBACCO                         store=64.7%  chain=93.4%  gap=-28.7pp  lsv=£341.79
  store= 476  dept=SANDWICHES  SNACKS              store=68.1%  chain=96.2%  gap=-28.1pp  lsv=£525.98
  store= 401  dept=TOBACCO                         store

## Step 5 — REPEAT Detector

In [34]:
from store_insight.pipeline.stages._1_detect.detectors.osa.repeat import detect as detect_repeat

repeat_signals = detect_repeat(
    qc_data,
    lookback_start=LOOKBACK_START,
    lookback_end=THIS_WEEK_END,
)

print(f'REPEAT signals: {len(repeat_signals)}')
for sig in sorted(repeat_signals, key=lambda x: x['repeat_item_count'], reverse=True)[:10]:
    print(f"  store={sig['store_id']:>4}  dept={sig['department']:<30}  items={sig['repeat_item_count']}  "
          f"lsv=£{sig['this_week_lsv']:,.2f}")
    for item in sig['repeat_item_details'][:3]:
        print(f"    • {item['ppg']:<40} {item['alert_weeks']} wks  "
              f"cum_lsv=£{item['cumulative_lsv']:,.2f}  avg_osa={item['avg_alert_osa']:.1f}%")

REPEAT signals: 321
  store= 213  dept=CIGARETTES                      items=15  lsv=£2,698.32
    • KINGSIZE STANDARD                        7 wks  cum_lsv=£1,459.16  avg_osa=60.4%
    • SUPERKINGS STANDARD                      7 wks  cum_lsv=£1,361.84  avg_osa=50.3%
    • KINGSIZE STANDARD                        8 wks  cum_lsv=£1,330.78  avg_osa=73.6%
  store= 430  dept=CIGARETTES                      items=15  lsv=£2,218.03
    • KINGSIZE STANDARD                        6 wks  cum_lsv=£1,398.47  avg_osa=41.8%
    • KINGSIZE MILD/LIGHTS                     5 wks  cum_lsv=£872.82  avg_osa=21.7%
    • KINGSIZE STANDARD                        4 wks  cum_lsv=£866.87  avg_osa=28.1%
  store= 215  dept=CIGARETTES                      items=11  lsv=£1,620.92
    • KINGSIZE STANDARD                        8 wks  cum_lsv=£1,355.91  avg_osa=77.4%
    • KINGSIZE MILD/LIGHTS                     6 wks  cum_lsv=£916.93  avg_osa=58.7%
    • KINGSIZE STANDARD                        6 wks  cum_lsv=£80

## Step 6 — TICK_AND_FLICK Detector

In [8]:
from store_insight.pipeline.stages._1_detect.detectors.compliance.tick_and_flick import detect as detect_tf

tf_signals = detect_tf(
    feedback,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    store_id=QC_STORE_ID,
)

print(f'TICK_AND_FLICK signals: {len(tf_signals)}')
for sig in sorted(tf_signals, key=lambda x: x['tf_rate_change'], reverse=True):
    print(
        f"  store={sig['store_id']:>4}  {sig.get('site_name', ''):<25}  "
        f"TW={sig['this_week_tf_rate']:.1f}%  LW={sig['last_week_tf_rate']:.1f}%  "
        f"change={sig['tf_rate_change']:+.1f}pp  "
        f"above_chain={sig['above_chain_by']:+.1f}pp  "
        f"worked={sig['this_week_worked']}"
    )

[1_detect] TICK_AND_FLICK=13 (chain avg 35.4%)
TICK_AND_FLICK signals: 13
  store= 411  Holbrook                   TW=64.8%  LW=0.0%  change=+64.8pp  above_chain=+29.4pp  worked=54
  store= 421  Capel St Mary              TW=62.2%  LW=0.0%  change=+62.2pp  above_chain=+26.8pp  worked=45
  store= 209  West Bergholt              TW=70.5%  LW=34.0%  change=+36.6pp  above_chain=+35.1pp  worked=95
  store= 407  Foxhall Road               TW=54.2%  LW=19.3%  change=+34.9pp  above_chain=+18.8pp  worked=96
  store= 440  Hasketon Road              TW=60.3%  LW=30.8%  change=+29.6pp  above_chain=+24.9pp  worked=63
  store= 448  Old Barrack Road           TW=61.5%  LW=32.3%  change=+29.2pp  above_chain=+26.1pp  worked=96
  store= 431  Wadgate Road               TW=59.2%  LW=33.8%  change=+25.4pp  above_chain=+23.8pp  worked=76
  store= 217  Wimpole Road               TW=59.6%  LW=42.7%  change=+16.9pp  above_chain=+24.2pp  worked=109
  store= 479  Lavenham                   TW=68.8%  LW=52.2%  ch

## Step 7 — WMD Data Load

Load and cache all WMD tables required by the markdown/waste detectors.  
Skipped if `RUN_WMD = False` in the config cell.

In [ ]:
if not RUN_WMD:
    print('Skipped — set RUN_WMD = True in config cell to run WMD steps.')
    markdown_log = markdown_sales = wastage = item_hierarchy = store_hierarchy = None
else:
    from snowflake.snowpark import functions as F, Window
    from snowflake.snowpark.types import LongType

    MARKDOWN_LOG_TABLE   = cfg.tables.markdown_log
    MARKDOWN_SALES_TABLE = cfg.tables.markdown_sales
    WASTAGE_TABLE        = cfg.tables.wastage
    WMD_ITEM_HIER_TABLE  = cfg.tables.wmd_item_hierarchy
    WMD_STORE_HIER_TABLE = cfg.tables.wmd_store_hierarchy

    print(f'Markdown log   : {MARKDOWN_LOG_TABLE}')
    print(f'Markdown sales : {MARKDOWN_SALES_TABLE}')
    print(f'Wastage        : {WASTAGE_TABLE}')
    print(f'Item hierarchy : {WMD_ITEM_HIER_TABLE}')
    print(f'Store hierarchy: {WMD_STORE_HIER_TABLE}')

    # WMD_MARKDOWN_API: SITE_ID→STORE_ID, MARKDOWN_DATE→LOCAL_DATE, MARKDOWN_QUANTITY→QTY_MARKDOWN
    markdown_log = (
        session.table(MARKDOWN_LOG_TABLE)
        .with_column('STORE_ID',     F.col('SITE_ID'))
        .with_column('LOCAL_DATE',   F.col('MARKDOWN_DATE'))
        .with_column('QTY_MARKDOWN', F.col('MARKDOWN_QUANTITY'))
        .filter(F.col('LOCAL_DATE') >= WMD_LOOKBACK)
        .filter(F.col('LOCAL_DATE') <= THIS_WEEK_END)
        .cache_result()
    )
    # WMD_SALES: SITE_ID→STORE_ID, SALE_QUANTITY→MARKDOWN_QUANTITY, SALE_VALUE→MARKDOWN_VALUE
    markdown_sales = (
        session.table(MARKDOWN_SALES_TABLE)
        .with_column('STORE_ID',          F.col('SITE_ID'))
        .with_column('MARKDOWN_QUANTITY', F.col('SALE_QUANTITY'))
        .with_column('MARKDOWN_VALUE',    F.col('SALE_VALUE'))
        .filter(F.col('DATE') >= WMD_LOOKBACK)
        .filter(F.col('DATE') <= THIS_WEEK_END)
        .cache_result()
    )
    # WMD_WASTAGE: SITE_ID→STORE_ID, WASTE_DATE→DATE
    wastage = (
        session.table(WASTAGE_TABLE)
        .with_column('STORE_ID', F.col('SITE_ID'))
        .with_column('DATE',     F.col('WASTE_DATE'))
        .filter(F.col('DATE') >= WMD_LOOKBACK)
        .filter(F.col('DATE') <= THIS_WEEK_END)
        .cache_result()
    )

    # WMD_ITEM_HIERARCHY: VME_CODE→ITEM_ID, ITEM_NAME→ITEM_DESCRIPTION, HIERARCHY_LEVEL_1_ID→DEPARTMENT
    _item_w = Window.partitionBy('VME_CODE').orderBy(F.col('FILE_TIMESTAMP').desc())
    item_hierarchy = (
        session.table(WMD_ITEM_HIER_TABLE)
        .with_column('_RN', F.row_number().over(_item_w))
        .filter(F.col('_RN') == 1)
        .select(
            F.col('VME_CODE').cast(LongType()).alias('ITEM_ID'),
            F.col('ITEM_NAME').alias('ITEM_DESCRIPTION'),
            F.col('HIERARCHY_LEVEL_1_ID').alias('DEPARTMENT'),
            F.col('SECTION'),
            F.col('SUB_SECTION'),
        )
        .filter(F.col('ITEM_ID').is_not_null())
        .cache_result()
    )

    # WMD_SITE_HIERARCHY: SITE_ID→STORE_ID, SITE_NAME→STORE_NAME
    store_hierarchy = (
        session.table(WMD_STORE_HIER_TABLE)
        .with_column('STORE_ID',   F.col('SITE_ID'))
        .with_column('STORE_NAME', F.col('SITE_NAME'))
        .cache_result()
    )

    print(f'\nMarkdown log rows  : {markdown_log.count():,}')
    print(f'Markdown sales rows: {markdown_sales.count():,}')
    print(f'Wastage rows       : {wastage.count():,}')
    print(f'Item hierarchy rows (deduplicated): {item_hierarchy.count():,}')
    print(f'Store hierarchy rows: {store_hierarchy.count():,}')

Markdown log   : INBOUND.SOURCE.SOURCE_WMD_MARKDOWN_LOG_CONVERTED
Markdown sales : INBOUND.SOURCE.SOURCE_WMD_MARKDOWN_SALES
Wastage        : INBOUND.SOURCE.SOURCE_WMD_WASTAGE
Item hierarchy : INBOUND.SOURCE.SOURCE_WMD_ITEM_HIERARCHY
Store hierarchy: INBOUND.SOURCE.SOURCE_WMD_STORE_HIERARCHY

Markdown log rows  : 280,822
Markdown sales rows: 619,354
Wastage rows       : 48,423
Item hierarchy rows (deduplicated): 179,660
Store hierarchy rows: 96,385


## Step 8 — MARKDOWN_OVERRIDE VS_CHAIN Detector

In [ ]:
if not RUN_WMD:
    print('Skipped.')
else:
    from store_insight.pipeline.stages._1_detect.detectors.markdown_override.vs_chain import detect as detect_md_override_vschain
    thresholds = cfg.detector_config.thresholds.model_dump()
    md_vc_signals = detect_md_override_vschain(
        markdown_log,
        store_hierarchy,
        this_week_start=THIS_WEEK_START,
        this_week_end=THIS_WEEK_END,
        min_override_chain_gap=thresholds.get('MARKDOWN_OVERRIDE', {}).get('min_override_chain_gap', 0.05),
        min_total_markdowns=thresholds.get('shared', {}).get('min_total_markdowns', 10),
        store_id=QC_STORE_ID,
    )
    for sig in sorted(md_vc_signals, key=lambda x: x['above_chain_pp'], reverse=True)[:15]:
        print(f"  store={sig['store_id']:>4}  {sig.get('site_name',''):<25}  "
              f"TW={sig['this_week_override_rate']:.1f}%  chain={sig['chain_override_rate']:.1f}%  "
              f"above={sig['above_chain_pp']:+.1f}pp  markdowns={sig['this_week_total_markdowns']}")

[1_detect] MARKDOWN_OVERRIDE VS_CHAIN=13 (chain avg 4.2%)
  store= 580  Harwich Road               TW=47.8%  chain=4.2%  above=+43.6pp  markdowns=454
  store= 407  Foxhall Road               TW=46.9%  chain=4.2%  above=+42.7pp  markdowns=793
  store= 411  Holbrook                   TW=34.0%  chain=4.2%  above=+29.8pp  markdowns=901
  store= 425  Barking Road               TW=28.7%  chain=4.2%  above=+24.5pp  markdowns=551
  store= 453  Beresford Road             TW=28.4%  chain=4.2%  above=+24.2pp  markdowns=373
  store= 234  Station Road               TW=23.5%  chain=4.2%  above=+19.3pp  markdowns=931
  store= 517  Acle Supermarket           TW=22.4%  chain=4.2%  above=+18.2pp  markdowns=1318
  store= 238  Connaught Avenue           TW=16.9%  chain=4.2%  above=+12.7pp  markdowns=794
  store= 494  Repps Road                 TW=15.3%  chain=4.2%  above=+11.2pp  markdowns=482
  store= 237  Holland                    TW=12.6%  chain=4.2%  above=+8.4pp  markdowns=374
  store= 634  Ipswich 

## Step 9 — MARKDOWN_OVERRIDE WOW Detector

In [ ]:
if not RUN_WMD:
    print('Skipped.')
else:
    from store_insight.pipeline.stages._1_detect.detectors.markdown_override.wow import detect as detect_md_override_wow
    thresholds = cfg.detector_config.thresholds.model_dump()
    md_wow_signals = detect_md_override_wow(
        markdown_log,
        store_hierarchy,
        last_week_start=LAST_WEEK_START,
        last_week_end=LAST_WEEK_END,
        this_week_start=THIS_WEEK_START,
        this_week_end=THIS_WEEK_END,
        min_override_wow_increase=thresholds.get('MARKDOWN_OVERRIDE', {}).get('min_override_wow_increase', 0.05),
        min_total_markdowns=thresholds.get('shared', {}).get('min_total_markdowns', 10),
        store_id=QC_STORE_ID,
    )
    for sig in sorted(md_wow_signals, key=lambda x: x['wow_change_pp'], reverse=True)[:15]:
        print(f"  store={sig['store_id']:>4}  {sig.get('site_name',''):<25}  "
              f"LW={sig['last_week_override_rate']:.1f}%  TW={sig['this_week_override_rate']:.1f}%  "
              f"change={sig['wow_change_pp']:+.1f}pp")

[1_detect] MARKDOWN_OVERRIDE WOW=6
  store= 234  Station Road               LW=7.4%  TW=23.5%  change=+16.1pp
  store= 411  Holbrook                   LW=18.7%  TW=34.0%  change=+15.3pp
  store= 634  Ipswich Station            LW=0.0%  TW=10.9%  change=+10.9pp
  store= 237  Holland                    LW=2.6%  TW=12.6%  change=+9.9pp
  store= 242  Fronks Road                LW=2.1%  TW=9.2%  change=+7.1pp
  store= 494  Repps Road                 LW=9.8%  TW=15.3%  change=+5.5pp


## Step 10 — WASTE VS_CHAIN Detector

In [ ]:
if not RUN_WMD:
    print('Skipped.')
else:
    from store_insight.pipeline.stages._1_detect.detectors.waste.vs_chain import detect as detect_waste_vschain
    thresholds = cfg.detector_config.thresholds.model_dump()
    waste_vc_signals = detect_waste_vschain(
        wastage,
        markdown_log,
        store_hierarchy,
        this_week_start=THIS_WEEK_START,
        this_week_end=THIS_WEEK_END,
        min_stb_chain_gap=thresholds.get('WASTE', {}).get('min_stb_chain_gap', 0.05),
        min_waste_qty=thresholds.get('shared', {}).get('min_waste_qty', 5),
        store_id=QC_STORE_ID,
    )
    for sig in sorted(waste_vc_signals, key=lambda x: x['above_chain_pp'], reverse=True)[:15]:
        print(f"  store={sig['store_id']:>4}  {sig.get('site_name',''):<25}  "
              f"waste_rate={sig['this_week_stb_rate']:.1f}%  chain={sig['chain_stb_rate']:.1f}%  "
              f"above={sig['above_chain_pp']:+.1f}pp  "
              f"waste_qty={sig['this_week_waste_qty']}  "
              f"val=£{sig['this_week_waste_value']:,.2f}")

[1_detect] WASTE VS_CHAIN=27 (chain avg 14.1%)
  store= 634  Ipswich Station            waste_rate=138.5%  chain=14.1%  above=+124.4pp  waste_qty=1550  val=£4,016.09
  store= 638  Chelmsford Station         waste_rate=96.5%  chain=14.1%  above=+82.4pp  waste_qty=412  val=£1,067.62
  store= 439  Walberswick                waste_rate=73.7%  chain=14.1%  above=+59.6pp  waste_qty=98  val=£161.11
  store= 492  Norwich Station            waste_rate=67.2%  chain=14.1%  above=+53.1pp  waste_qty=129  val=£334.69
  store= 406  Springfield                waste_rate=53.7%  chain=14.1%  above=+39.6pp  waste_qty=110  val=£196.92
  store= 421  Capel St Mary              waste_rate=44.3%  chain=14.1%  above=+30.2pp  waste_qty=156  val=£334.14
  store= 431  Wadgate Road               waste_rate=35.7%  chain=14.1%  above=+21.6pp  waste_qty=110  val=£152.26
  store= 440  Hasketon Road              waste_rate=34.5%  chain=14.1%  above=+20.4pp  waste_qty=329  val=£176.72
  store= 442  Woolpit              

In [13]:
if RUN_WMD:
    from snowflake.snowpark import functions as F
    # ── Diagnose: why is chain STB rate 100%? ────────────────────────────────
    # Check FINAL_PRICE nullability — if always NULL, STB = total waste = 100%
    diag = (
        wastage
        .filter(
            (F.col('DATE') >= THIS_WEEK_START) &
            (F.col('DATE') <= THIS_WEEK_END)
        )
        .agg(
            F.count('*').alias('TOTAL_ROWS'),
            F.sum(F.iff(F.col('FINAL_PRICE').is_null(), F.lit(1), F.lit(0))).alias('FINAL_PRICE_NULL'),
            F.sum(F.iff(F.col('FINAL_PRICE').is_not_null(), F.lit(1), F.lit(0))).alias('FINAL_PRICE_SET'),
            F.sum('WASTE_QUANTITY').alias('TOTAL_WASTE_QTY'),
        )
    ).collect()[0]

    print('WASTAGE this week diagnostics:')
    print(f'  Total rows          : {diag["TOTAL_ROWS"]:,}')
    print(f'  FINAL_PRICE = NULL  : {diag["FINAL_PRICE_NULL"]:,}  ({diag["FINAL_PRICE_NULL"]/max(diag["TOTAL_ROWS"],1)*100:.1f}%)')
    print(f'  FINAL_PRICE not NULL: {diag["FINAL_PRICE_SET"]:,}  ({diag["FINAL_PRICE_SET"]/max(diag["TOTAL_ROWS"],1)*100:.1f}%)')
    print(f'  Total waste qty     : {diag["TOTAL_WASTE_QTY"]:,}')
    print()
    print('Sample rows with FINAL_PRICE populated:')
    wastage.filter(
        (F.col('DATE') >= THIS_WEEK_START) &
        (F.col('DATE') <= THIS_WEEK_END) &
        F.col('FINAL_PRICE').is_not_null()
    ).select('STORE_ID','ITEM_ID','DATE','WASTE_QUANTITY','ORIGINAL_PRICE','FINAL_PRICE','WASTE_REASON').show(5)

WASTAGE this week diagnostics:
  Total rows          : 11,084
  FINAL_PRICE = NULL  : 11,084  (100.0%)
  FINAL_PRICE not NULL: 0  (0.0%)
  Total waste qty     : 27,122.5200

Sample rows with FINAL_PRICE populated:
----------------------------------------------------------------------------------------------------------
|"STORE_ID"  |"ITEM_ID"  |"DATE"  |"WASTE_QUANTITY"  |"ORIGINAL_PRICE"  |"FINAL_PRICE"  |"WASTE_REASON"  |
----------------------------------------------------------------------------------------------------------
|            |           |        |                  |                  |               |                |
----------------------------------------------------------------------------------------------------------



## Step 11 — WASTE WOW Detector

In [14]:
if not RUN_WMD:
    print('Skipped.')
else:
    from store_insight.pipeline.stages._1_detect.detectors.waste.wow import detect as detect_waste_wow
    thresholds = cfg.get('thresholds', {})
    waste_wow_signals = detect_waste_wow(
        wastage,
        markdown_log,
        store_hierarchy,
        last_week_start=LAST_WEEK_START,
        last_week_end=LAST_WEEK_END,
        this_week_start=THIS_WEEK_START,
        this_week_end=THIS_WEEK_END,
        min_stb_wow_increase=thresholds.get('min_stb_wow_increase', 0.05),
        min_waste_qty=thresholds.get('min_waste_qty', 5),
        store_id=QC_STORE_ID,
    )
    for sig in sorted(waste_wow_signals, key=lambda x: x['wow_change_pp'], reverse=True)[:15]:
        lw = sig['last_week_stb_rate']
        lw_str = f'{lw:.1f}%' if lw is not None else '—'
        print(f"  store={sig['store_id']:>4}  {sig.get('site_name',''):<25}  "
              f"LW={lw_str}  TW={sig['this_week_stb_rate']:.1f}%  "
              f"change={sig['wow_change_pp']:+.1f}pp  waste_qty={sig['this_week_waste_qty']}")

[1_detect] WASTE WOW=32
  store= 638  Chelmsford Station         LW=39.1%  TW=96.5%  change=+57.4pp  waste_qty=412
  store= 439  Walberswick                LW=18.0%  TW=73.7%  change=+55.7pp  waste_qty=98
  store= 634  Ipswich Station            LW=88.8%  TW=138.5%  change=+49.7pp  waste_qty=1550
  store= 421  Capel St Mary              LW=—  TW=44.3%  change=+44.3pp  waste_qty=156
  store= 406  Springfield                LW=17.0%  TW=53.7%  change=+36.7pp  waste_qty=110
  store= 492  Norwich Station            LW=36.9%  TW=67.2%  change=+30.3pp  waste_qty=129
  store= 440  Hasketon Road              LW=9.4%  TW=34.5%  change=+25.1pp  waste_qty=329
  store= 431  Wadgate Road               LW=13.6%  TW=35.7%  change=+22.1pp  waste_qty=110
  store= 442  Woolpit                    LW=17.6%  TW=34.2%  change=+16.6pp  waste_qty=195
  store= 215  Shrub End                  LW=20.2%  TW=32.6%  change=+12.4pp  waste_qty=356
  store= 204  West Mersea Supermarket    LW=9.7%  TW=21.7%  change=+12

## Step 12 — MARKDOWN_SELLTHROUGH VS_CHAIN Detector

In [15]:
wastage.select('WASTE_REASON').distinct().show(20)

--------------------------
|"WASTE_REASON"          |
--------------------------
|TOO GOOD TO GO          |
|OUT OF DATE             |
|DAMAGED                 |
|AFTER BEST BEFORE DATE  |
--------------------------



In [16]:
if not RUN_WMD:
    print('Skipped.')
else:
    from store_insight.pipeline.stages._1_detect.detectors.markdown_sellthrough.vs_chain import detect as detect_st_vschain
    thresholds = cfg.get('thresholds', {})
    st_vc_signals = detect_st_vschain(
        markdown_sales,
        markdown_log,
        store_hierarchy,
        this_week_start=THIS_WEEK_START,
        this_week_end=THIS_WEEK_END,
        min_sellthrough_chain_gap=thresholds.get('min_sellthrough_chain_gap', 0.05),
        min_markdown_qty=thresholds.get('min_markdown_qty', 10),
        store_id=QC_STORE_ID,
    )
    for sig in sorted(st_vc_signals, key=lambda x: x['below_chain_pp'], reverse=True)[:15]:
        print(f"  store={sig['store_id']:>4}  {sig.get('site_name',''):<25}  "
              f"ST={sig['this_week_sellthrough_rate']:.1f}%  chain={sig['chain_sellthrough_rate']:.1f}%  "
              f"below={sig['below_chain_pp']:+.1f}pp  md_qty={sig['this_week_markdown_qty']}")

[1_detect] MARKDOWN_SELLTHROUGH VS_CHAIN=56 (chain avg 99.1%)
  store= 480  LE CHATEAU (Old store)     ST=0.0%  chain=99.1%  below=+99.1pp  md_qty=519
  store= 520  Witham                     ST=52.2%  chain=99.1%  below=+46.9pp  md_qty=874
  store= 270  Burnham Supermarket        ST=53.7%  chain=99.1%  below=+45.4pp  md_qty=8522
  store= 580  Harwich Road               ST=61.5%  chain=99.1%  below=+37.7pp  md_qty=1004
  store= 540  Lawling Avenue             ST=61.9%  chain=99.1%  below=+37.2pp  md_qty=793
  store= 404  Great Cornard              ST=63.2%  chain=99.1%  below=+35.9pp  md_qty=1324
  store= 497  Riverside                  ST=65.2%  chain=99.1%  below=+33.9pp  md_qty=1625
  store= 407  Foxhall Road               ST=67.2%  chain=99.1%  below=+31.9pp  md_qty=1726
  store= 465  Mile Cross                 ST=67.2%  chain=99.1%  below=+31.9pp  md_qty=985
  store= 511  Cannerby Lane              ST=68.2%  chain=99.1%  below=+30.9pp  md_qty=1112
  store= 433  Chantry            

## Step 13 — MARKDOWN_SELLTHROUGH WOW Detector

In [17]:
if not RUN_WMD:
    print('Skipped.')
else:
    from store_insight.pipeline.stages._1_detect.detectors.markdown_sellthrough.wow import detect as detect_st_wow
    thresholds = cfg.get('thresholds', {})
    st_wow_signals = detect_st_wow(
        markdown_sales,
        markdown_log,
        store_hierarchy,
        last_week_start=LAST_WEEK_START,
        last_week_end=LAST_WEEK_END,
        this_week_start=THIS_WEEK_START,
        this_week_end=THIS_WEEK_END,
        min_sellthrough_wow_drop=thresholds.get('min_sellthrough_wow_drop', 0.05),
        min_markdown_qty=thresholds.get('min_markdown_qty', 10),
        store_id=QC_STORE_ID,
    )
    for sig in sorted(st_wow_signals, key=lambda x: x['wow_change_pp'])[:15]:
        print(f"  store={sig['store_id']:>4}  {sig.get('site_name',''):<25}  "
              f"LW={sig['last_week_sellthrough_rate']:.1f}%  TW={sig['this_week_sellthrough_rate']:.1f}%  "
              f"change={sig['wow_change_pp']:+.1f}pp  md_qty={sig['this_week_markdown_qty']}")

[1_detect] MARKDOWN_SELLTHROUGH WOW=5
  store= 270  Burnham Supermarket        LW=88.3%  TW=53.7%  change=-34.6pp  md_qty=8522
  store= 580  Harwich Road               LW=73.0%  TW=61.5%  change=-11.5pp  md_qty=1004
  store= 210  Dedham                     LW=83.4%  TW=73.5%  change=-9.9pp  md_qty=1059
  store= 460  Hall Road                  LW=120.6%  TW=110.7%  change=-9.8pp  md_qty=1549
  store= 248  Silver End                 LW=91.5%  TW=86.0%  change=-5.5pp  md_qty=1537


## Step 14 — MARKDOWN_SKU REPEAT_REQUEST Detector

## Step 15 — MARKDOWN_SKU EXCESSIVE_OVERRIDE Detector

In [18]:
if not RUN_WMD:
    print('Skipped.')
else:
    from store_insight.pipeline.stages._1_detect.detectors.markdown_sku.excessive_override import detect as detect_excess_override
    thresholds = cfg.get('thresholds', {})
    eo_signals = detect_excess_override(
        markdown_log,
        item_hierarchy,
        store_hierarchy,
        window_end=THIS_WEEK_END,
        lookback_weeks=thresholds.get('lookback_weeks', 4),
        min_override_rate=thresholds.get('min_sku_override_rate', 0.50),
        min_requests=thresholds.get('min_sku_requests', 3),
        store_id=QC_STORE_ID,
    )
    print(f'EXCESSIVE_OVERRIDE signals: {len(eo_signals)}')
    for sig in sorted(eo_signals,
                      key=lambda x: (x['override_count'] or 0) / max(x['markdown_request_count'], 1),
                      reverse=True)[:15]:
        rate = round((sig['override_count'] or 0) / max(sig['markdown_request_count'], 1) * 100)
        print(f"  store={sig['store_id']:>4}  item={sig['item_id']:>8}  "
              f"{sig.get('item_description',''):<35}  "
              f"overrides={sig['override_count']}/{sig['markdown_request_count']} ({rate}%)")

[1_detect] MARKDOWN_SKU EXCESSIVE_OVERRIDE=455 (lookback=4w, min_rate=50%)
EXCESSIVE_OVERRIDE signals: 455
  store= 505  item=    4063                                       overrides=2/3 (67%)
  store= 462  item=8000300430776                                       overrides=2/3 (67%)
  store= 292  item=8445290161147                                       overrides=2/3 (67%)
  store= 238  item=5000128584807                                       overrides=4/6 (67%)
  store= 535  item=5000129321197                                       overrides=2/3 (67%)
  store= 442  item=      78                                       overrides=2/3 (67%)
  store= 505  item=7640175744267                                       overrides=4/6 (67%)
  store= 438  item=5000328839400                                       overrides=2/3 (67%)
  store= 237  item=5000128779586                                       overrides=5/8 (62%)
  store= 407  item=5000129351972                                       overrides=5/8

## Step 16 — Merged Signals (all detectors)

In [19]:
from store_insight.pipeline.stages._1_detect.detect import detect_signals
from collections import Counter

wmd_kwargs = {}
if RUN_WMD:
    wmd_kwargs = dict(
        markdown_log=markdown_log,
        markdown_sales=markdown_sales,
        wastage=wastage,
        item_hierarchy=item_hierarchy,
        store_hierarchy=store_hierarchy,
    )

signals = detect_signals(
    qc_data,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    feedback=feedback,
    cfg=cfg,
    **wmd_kwargs,
)

trigger_counts = Counter(tuple(sorted(s['trigger_types'])) for s in signals)
type_counts    = Counter(s['signal_type'] for s in signals)

print(f'Total signals: {len(signals)}')
print(f'\nBy signal_type:')
for stype, count in sorted(type_counts.items()):
    print(f'  {stype:<25} {count}')

print(f'\nBy trigger combination:')
for triggers, count in sorted(trigger_counts.items()):
    print(f'  {"+".join(triggers):<40} {count}')

multi = [s for s in signals if len(s['trigger_types']) > 1]
print(f'\nMulti-trigger signals: {len(multi)}')
for s in multi[:5]:
    label = s.get('department') or s.get('item_description') or ''
    print(f"  store={s['store_id']:>4}  type={s['signal_type']:<20}  {label:<35}  {s['trigger_types']}")

[1_detect] TICK_AND_FLICK=13 (chain avg 35.4%)
[1_detect] MARKDOWN_OVERRIDE VS_CHAIN=13 (chain avg 4.2%)
[1_detect] MARKDOWN_OVERRIDE WOW=6
[1_detect] MARKDOWN_SELLTHROUGH VS_CHAIN=56 (chain avg 99.1%)
[1_detect] MARKDOWN_SELLTHROUGH WOW=5
[1_detect] MARKDOWN_SKU EXCESSIVE_OVERRIDE=455 (lookback=4w, min_rate=50%)
[1_detect] MARKDOWN_SKU REPEAT_REQUEST=18365 (lookback=4w, min_weeks=3)
[1_detect] WASTE VS_CHAIN=27 (chain avg 14.1%)
[1_detect] WASTE WOW=32
[1_detect] 19180 signals — TICK_AND_FLICK=13, VS_CHAIN=27, WOW=32, EXCESSIVE_OVERRIDE=455, REPEAT_REQUEST=18365, REPEAT=321
Total signals: 19180

By signal_type:
  COMPLIANCE                13
  MARKDOWN_OVERRIDE         13
  MARKDOWN_SELLTHROUGH      57
  MARKDOWN_SKU              18486
  OSA                       574
  WASTE                     37

By trigger combination:
  EXCESSIVE_OVERRIDE                       345
  EXCESSIVE_OVERRIDE+REPEAT_REQUEST        109
  REPEAT                                   228
  REPEAT+VS_CHAIN       